# Example code for connecting a Sciospec EIT device

This script gives an example how to use this updated version of sciopy.   
  
Central new feature is an automated USB messaage parser, that parses incoming messages immediately and upon request saves data frames while running. For burst measurements AND continuous measurements with burst_count = 0

In [ ]:
# Initialization
import matplotlib.pyplot as plt
import numpy as np
import time
from sciopy.sciopy import EIT_16_32_64_128, EitMeasurementSetup, available_serial_ports
from sciopy.sciopy import make_results_folder
# create a 'sciospec' EIT device
n_el = 16
sciospec = EIT_16_32_64_128(n_el)
# connect device via USB-FS port
print(available_serial_ports())
sciospec.connect_device_FS("COM3")
savepath = "../../data/"

In [ ]:
# read system message buffer
sciospec.SystemMessageCallback()

In [ ]:
# create a measurement setup
setup = EitMeasurementSetup(
    burst_count=4,
    n_el=n_el,
    exc_freq=125_000,
    framerate=3,
    amplitude=0.01,
    inj_skip=n_el // 2,
    gain=1,
    adc_range=1
)
sciospec.SetMeasurementSetup(setup)
# look inside the docstring of the function and manual
sciospec.GetMeasurementSetup(2)

### Usage of the "old" measure data command 

Here, the message parser is not utilized, technically faster processing of incoming messages. Messages are solely appended and processed into EITFrames and bursts

In [ ]:
data = sciospec.StartStopMeasurementFast(return_as="hex")
print(data.shape)


### New measure-data function


In [ ]:
# 1) With burstcount
sciospec.update_BurstCount(3)   
data= sciospec.StartStopMeasurement(return_as="eitframe", bSaveData=False, bDeleteData=False,
                                  sSavePath=savepath, bResultsFolder=False)
print(data.shape)

# 2) Continuous measurement
sciospec.update_BurstCount(0)
data= sciospec.StartStopMeasurement(timeout= 5,return_as="eitframe", bSaveData=False, bDeleteData=False,
                                  sSavePath=savepath,bResultsFolder=False) # measured for five seconds
print(data.shape)



# 3) Save data in real time and create an additional folder for storage
data= sciospec.StartStopMeasurement(timeout= 5,return_as="eitframe", bSaveData=True, bDeleteData=False,
                                  sSavePath=savepath,bResultsFolder=True) # measured for five seconds
print(data.shape)




# 4) For continuous saving results in the same folder, create a folder before and then pass it along
sCurrPath= make_results_folder(bCreateResultsFolder=True, bSaveData=True, sSavePath=savepath)
sciospec.StartStopMeasurement(timeout= 5,return_as="eitframe", bSaveData=True, bDeleteData=True,
                                  sSavePath=sCurrPath,bResultsFolder=False) # measured for five seconds
time.sleep(3)
sciospec.StartStopMeasurement(timeout= 5,return_as="eitframe", bSaveData=True, bDeleteData=True,
                                  sSavePath=sCurrPath,bResultsFolder=False) # measured for five seconds
time.sleep(3)
sciospec.StartStopMeasurement(timeout= 5,return_as="eitframe", bSaveData=True, bDeleteData=True,
                                  sSavePath=sCurrPath,bResultsFolder=False) # measured for five seconds



# Arguments:
# bDeleteData: Data is not returned but if bSaveData=True saved, and is deleted from an internal buffer. For False, data is returned according to return_as
# bSaveData: if Data should be saved in-time with the measurements. Data is saved at sSavePath, 


### Example on how to change the measurement mode with boundary conditions is updated

Recommended is for every change of the measurement mode
1. Restart Device
2. Set new measurement mode

In [ ]:
# Standard setting "singleended" -> potential measurement
sciospec.GetMeasurementSetup(2)
sciospec.update_BurstCount(3)   
data= sciospec.StartStopMeasurement(return_as="eitframe", bSaveData=False, bDeleteData=False,
                                  sSavePath=savepath, bResultsFolder=False)
data_pot=np.abs(data[2])



In [ ]:
# For updated setting, restart device
sciospec.SoftwareReset()
sciospec = EIT_16_32_64_128(16)
print(available_serial_ports())
sciospec.connect_device_FS("COM3")
sciospec.SetMeasurementSetup(setup)
sciospec.update_measurement_mode("skip4", "internal")
sciospec.GetMeasurementSetup(2)
# look inside the docstring of the function and manual

sciospec.update_BurstCount(3)   
data= sciospec.StartStopMeasurement(return_as="eitframe", bSaveData=False, bDeleteData=False,
                                  sSavePath=savepath, bResultsFolder=False)
data_skip4= np.abs(data[2])


fig, ax = plt.subplots(ncols=2)
ax[0].imshow(data_pot, cmap="viridis")
ax[0].set_title("Singleended")
ax[1].imshow(data_skip4, cmap="viridis")
ax[1].set_title("Skip4")
plt.tight_layout()
plt.show()

In [ ]:

# Alternatively,
setup = EitMeasurementSetup(
    burst_count=4,
    n_el=n_el,
    exc_freq=125_000,
    framerate=3,
    amplitude=0.01,
    inj_skip=n_el // 2,
    gain=1,
    adc_range=1,
    mea_mode="skip2",
    mea_mode_boundary="external"
)
sciospec.SetMeasurementSetup(setup)



